# DDoS mitigation on FABRIC

This notebook deploys the mitigation half of the ML-DDoS system on a FABRIC slice and runs
it against a controlled flood:

1. build a slice: one switch node (Open vSwitch) and five hosts on point-to-point L2 links
2. install the runtime: OpenFlow 1.3 controller, flow meter, traffic monitor and the
   **already trained** Random Forest (no training happens here)
3. run an attack with mitigation **on**, watching the drop rules appear and count packets
4. optionally run the same attack with mitigation **off**, for comparison
5. compute the metrics and figures, then delete the slice

```
                     switch   Open vSwitch br0 + controller + monitor
               /        |         |         \         \
            srv        h1        h3         h2        h4
        web service  attacker  attacker   client    client
        10.0.0.100   10.0.0.1  10.0.0.3   10.0.0.2  10.0.0.4
```

The monitor taps the switch port of the server link, turns packets into flow features,
scores them with the model, and asks the controller to install a temporary drop rule for
a source once at least 5 of its flows in 6 s are classified as attack and they are at
least 80 % of its traffic, or once it produces more than 100 flows per second. The
controller renews a rule while it is still dropping packets and lets it expire afterwards.

Run the cells in order. Every step can be re-run; after a kernel restart, run the
*Set up* cells and then `tb.attach()` instead of `tb.create()`.

## 1 · Set up

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()                      # this notebook's folder: <repository>/fabric
sys.path.insert(0, str(REPO))

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
from fabric_ddos.testbed import Config, Testbed
from fabric_ddos import analysis

fablib = fablib_manager()
fablib.show_config();

In [ ]:
cfg = Config(
    slice_name="ddos-mitigation",
    site=None,            # None = a random site with free resources; or e.g. "TACC"
    n_attackers=2,
    n_clients=2,
    block_seconds=20,     # lifetime of a drop rule before it is renewed or expires
    server_rate_mbit=20,  # the server link is shaped so an unmitigated flood hurts
    host_rate_mbit=10,
)
tb = Testbed(fablib, cfg)
[(h.name, h.ip, h.role) for h in tb.hosts]

## 2 · Create the slice

Takes a few minutes. If the slice already exists, use `tb.attach()` instead.

In [ ]:
tb.create()        # or: tb.attach()

## 3 · Install the runtime

Uploads the controller, flow meter, monitor, traffic tools and trained model from this
repository, plus the node scripts in `runtime/`, and installs the dependencies: Open vSwitch, tcpdump and
two Python environments on the switch node (pinned scikit-learn for the model, os-ken for
the controller), Flask on the hosts. The last line per node confirms the install; for the
switch node it also confirms that the model loads.

In [ ]:
tb.install()

## 4 · Configure the data plane

Addresses and shaping on the hosts, the OpenFlow bridge on the switch node, and the
forwarding table. Forwarding is by destination IP with the MAC addresses rewritten, because
the switch ports are SR-IOV virtual functions that only accept their own MAC address.

In [ ]:
tb.configure_network()

## 5 · Attack with mitigation on

Benign clients run for the whole scenario; the UDP flood from `h1` and `h3` starts after 10 s.
While it runs, each line shows what the controller reports: the active drop rules, how many
packets per second each is dropping, and how long until it expires or is renewed.

In [ ]:
run_mit = tb.run_scenario("attack_mit", mitigation=True, duration=45, attack_delay=10)

## 6 · The same attack with mitigation off (optional)

The detector still runs and logs its decisions, but the controller installs nothing, so
the flood reaches the server. This is the baseline that shows what mitigation prevents.

In [ ]:
run_nomit = tb.run_scenario("attack_nomit", mitigation=False, duration=45, attack_delay=10)

## 7 · Results

Detection time is measured from the first attack packet seen at the tap (resolution: one
0.5 s telemetry window). Dropped packets come from the drop rules' own counters.

In [ ]:
from IPython.display import Markdown, display

run_dirs = [d for d in (REPO / "results/attack_nomit", REPO / "results/attack_mit") if (d / "scenario.json").exists()]
runs = [analysis.load_run(d) for d in run_dirs]
summaries = [analysis.summarize(r) for r in runs]
display(Markdown(analysis.summary_table(summaries)))

In [ ]:
fig = analysis.plot_runs(runs)
fig.savefig(REPO / "results/mitigation.png", dpi=160)

The collected run directories use the same file format as the Mininet testbed, so they can
be replayed in the ML-DDoS dashboard on your own machine:

```bash
uv run python monitor/server/app.py --mode replay --run fabric/results/attack_mit
```

## 8 · Try it by hand (optional)

The controller API is reachable from the notebook through the switch node.

In [ ]:
tb.status()                      # switches, mitigation flag, active drop rules and their counters
# tb.block("10.0.0.1")           # install a drop rule by hand
# tb.unblock("10.0.0.1")
# tb.set_mitigation(False)       # detect only

## 9 · Clean up

Deletes the slice and releases its resources. Results stay in `results/`.

In [ ]:
tb.stop()
tb.delete()